# Phase 4: Sparse Indexing (BM25)

**Pipeline**: Vietnamese Financial News RAG System — v3  
**Owner**: Member A | **Hardware**: Colab CPU  
**Library**: `rank-bm25` (BM25Okapi)

### What this notebook does
For **each of the 3 chunking strategies** (`fixed_size`, `sentence_aware`, `article_level`):
1. Load `data/chunks/{strategy}/chunks.parquet`
2. Tokenise every chunk text with the whitespace Vietnamese tokenizer
3. Build a `BM25Okapi` index
4. Persist to `bm25/{strategy}/bm25_index.pkl` + `chunk_ids.json` + `build_stats.json`
5. Verify alignment: BM25 corpus length == len(chunk_ids)

All cells are **idempotent** — if an index already exists it is loaded from cache, not rebuilt.

> ⏱️ **Expected time**: 5–10 min per strategy on Colab CPU. All 3 run sequentially in this notebook.

## Cell 0 — Environment Setup

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path

# ── Detect Colab vs local ──────────────────────────────────────────────────
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Runtime: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    GIT_DIR = Path('/content/rag-vn-finance')

    # Chỉ cài đặt và khởi động lại Kernel 1 lần duy nhất
    if not GIT_DIR.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(GIT_DIR)])

        req_path = GIT_DIR / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')

        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel để nạp đồng bộ Numpy/Pandas...")
        os.kill(os.getpid(), 9) # Ép Colab khởi động lại RAM
    else:
        print("Mã nguồn đã tồn tại. Đang cập nhật code mới nhất từ Github...")
        subprocess.run(['git', '-C', str(GIT_DIR), 'pull'], capture_output=True)

    REPO_ROOT = GIT_DIR
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Cell 0 complete.")

Runtime: Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mã nguồn đã tồn tại. Đang cập nhật code mới nhất từ Github...
Project root: /content/rag-vn-finance
Cell 0 complete.


## Cell 1 — Load Config & Resolve Paths

In [2]:
import json
import pandas as pd
from src.utils import load_config, resolve_path

config   = load_config(REPO_ROOT / 'configs' / 'config.yaml')
strategies = config['chunking']['strategies']

# ── Chunks input base dir ───────────────────────────────────────────────────
chunks_base = resolve_path(config['chunking'], 'output_dir')
if not os.path.isabs(chunks_base):
    chunks_base = str(REPO_ROOT / chunks_base)

# ── BM25 output base dir ────────────────────────────────────────────────────
bm25_base = resolve_path(config['indexing'], 'bm25_dir')
if not os.path.isabs(bm25_base):
    bm25_base = str(REPO_ROOT / bm25_base)

print(f"Strategies      : {strategies}")
print(f"Chunks input    : {chunks_base}")
print(f"BM25 output dir : {bm25_base}")

# Verify all chunk files exist before starting
for s in strategies:
    p = os.path.join(chunks_base, s, 'chunks.parquet')
    status = '✅' if os.path.exists(p) else '❌ MISSING'
    print(f"  [{s}] {status}")

print("\nCell 1 complete.")

Strategies      : ['fixed_size', 'sentence_aware', 'article_level']
Chunks input    : /content/drive/MyDrive/rag-vn-finance/data/chunks
BM25 output dir : /content/drive/MyDrive/rag-vn-finance/bm25
  [fixed_size] ✅
  [sentence_aware] ✅
  [article_level] ✅

Cell 1 complete.


## Cell 2 — Tokenizer Sanity Check

> **Design decision**: We use a whitespace-based Vietnamese tokenizer as the  
> baseline. Vietnamese compound words (e.g., `ngân hàng`) are split into  
> unigrams. This is a known limitation documented in the Phase 10 report.  
> `underthesea` word segmentation is an optional upgrade for future work.

In [3]:
from src.indexing import tokenize_vi

test_cases = [
    "Ngân hàng Nhà nước Việt Nam tăng lãi suất điều hành lên 6%.",
    "Cổ phiếu VIC tăng mạnh trong phiên giao dịch hôm nay.",
    "GDP Việt Nam năm 2023 đạt 5,05% so với năm trước.",
    "",                   # edge case: empty string
    "a b c",             # edge case: all single-char tokens (should be dropped)
]

print("Tokenizer output:")
for t in test_cases:
    tokens = tokenize_vi(t)
    print(f"  Input : {t[:60]!r}")
    print(f"  Tokens: {tokens[:10]} ({len(tokens)} total)")
    print()

print("Cell 2 complete.")

Tokenizer output:
  Input : 'Ngân hàng Nhà nước Việt Nam tăng lãi suất điều hành lên 6%.'
  Tokens: ['ngân', 'hàng', 'nhà', 'nước', 'việt', 'nam', 'tăng', 'lãi', 'suất', 'điều'] (12 total)

  Input : 'Cổ phiếu VIC tăng mạnh trong phiên giao dịch hôm nay.'
  Tokens: ['cổ', 'phiếu', 'vic', 'tăng', 'mạnh', 'trong', 'phiên', 'giao', 'dịch', 'hôm'] (11 total)

  Input : 'GDP Việt Nam năm 2023 đạt 5,05% so với năm trước.'
  Tokens: ['gdp', 'việt', 'nam', 'năm', '2023', 'đạt', '05', 'so', 'với', 'năm'] (11 total)

  Input : ''
  Tokens: [] (0 total)

  Input : 'a b c'
  Tokens: [] (0 total)

Cell 2 complete.


## Cell 3 — Build BM25 Index: `fixed_size`

> ⏱️ Expected: ~5–10 min on Colab CPU.  
> If `bm25_index.pkl` already exists, this cell loads it from cache instantly.

In [4]:
from src.indexing import build_bm25_index

STRATEGY = 'fixed_size'
chunks_path = os.path.join(chunks_base, STRATEGY, 'chunks.parquet')
output_dir  = os.path.join(bm25_base,   STRATEGY)

print(f"Loading chunks: {chunks_path}")
df_fs = pd.read_parquet(chunks_path)
print(f"Chunks loaded: {len(df_fs):,}")

bm25_fs, ids_fs = build_bm25_index(df_fs, output_dir, STRATEGY)

stats_path = os.path.join(output_dir, 'build_stats.json')
if os.path.exists(stats_path):
    print("\nBuild stats:")
    print(json.dumps(json.load(open(stats_path)), indent=2))

print(f"\n✅ fixed_size BM25 ready — {len(ids_fs):,} chunks, avgdl={bm25_fs.avgdl:.1f} tokens")

Loading chunks: /content/drive/MyDrive/rag-vn-finance/data/chunks/fixed_size/chunks.parquet


[2026-05-11 06:50:05] [INFO] src.indexing: [fixed_size] BM25 index already exists — loading from cache.
INFO:src.indexing:[fixed_size] BM25 index already exists — loading from cache.


Chunks loaded: 45,764


[2026-05-11 06:50:07] [INFO] src.indexing: [fixed_size] Loaded: 45,764 chunks.
INFO:src.indexing:[fixed_size] Loaded: 45,764 chunks.



Build stats:
{
  "strategy": "fixed_size",
  "total_chunks": 45764,
  "vocab_size": 37546,
  "avg_doc_length_tokens": 206.0,
  "tokenization_seconds": 6.2,
  "build_seconds": 4.3,
  "tokenizer": "whitespace_vi",
  "bm25_variant": "BM25Okapi"
}

✅ fixed_size BM25 ready — 45,764 chunks, avgdl=206.0 tokens


## Cell 4 — Build BM25 Index: `sentence_aware`

In [5]:
STRATEGY = 'sentence_aware'
chunks_path = os.path.join(chunks_base, STRATEGY, 'chunks.parquet')
output_dir  = os.path.join(bm25_base,   STRATEGY)

print(f"Loading chunks: {chunks_path}")
df_sa = pd.read_parquet(chunks_path)
print(f"Chunks loaded: {len(df_sa):,}")

bm25_sa, ids_sa = build_bm25_index(df_sa, output_dir, STRATEGY)

stats_path = os.path.join(output_dir, 'build_stats.json')
if os.path.exists(stats_path):
    print("\nBuild stats:")
    print(json.dumps(json.load(open(stats_path)), indent=2))

print(f"\n✅ sentence_aware BM25 ready — {len(ids_sa):,} chunks, avgdl={bm25_sa.avgdl:.1f} tokens")

Loading chunks: /content/drive/MyDrive/rag-vn-finance/data/chunks/sentence_aware/chunks.parquet


[2026-05-11 06:50:08] [INFO] src.indexing: [sentence_aware] BM25 index already exists — loading from cache.
INFO:src.indexing:[sentence_aware] BM25 index already exists — loading from cache.


Chunks loaded: 64,197


[2026-05-11 06:50:10] [INFO] src.indexing: [sentence_aware] Loaded: 64,197 chunks.
INFO:src.indexing:[sentence_aware] Loaded: 64,197 chunks.



Build stats:
{
  "strategy": "sentence_aware",
  "total_chunks": 64197,
  "vocab_size": 36957,
  "avg_doc_length_tokens": 153.4,
  "tokenization_seconds": 4.6,
  "build_seconds": 5.7,
  "tokenizer": "whitespace_vi",
  "bm25_variant": "BM25Okapi"
}

✅ sentence_aware BM25 ready — 64,197 chunks, avgdl=153.4 tokens


## Cell 5 — Build BM25 Index: `article_level`

In [6]:
STRATEGY = 'article_level'
chunks_path = os.path.join(chunks_base, STRATEGY, 'chunks.parquet')
output_dir  = os.path.join(bm25_base,   STRATEGY)

print(f"Loading chunks: {chunks_path}")
df_al = pd.read_parquet(chunks_path)
print(f"Chunks loaded: {len(df_al):,}")

bm25_al, ids_al = build_bm25_index(df_al, output_dir, STRATEGY)

stats_path = os.path.join(output_dir, 'build_stats.json')
if os.path.exists(stats_path):
    print("\nBuild stats:")
    print(json.dumps(json.load(open(stats_path)), indent=2))

print(f"\n✅ article_level BM25 ready — {len(ids_al):,} chunks, avgdl={bm25_al.avgdl:.1f} tokens")

Loading chunks: /content/drive/MyDrive/rag-vn-finance/data/chunks/article_level/chunks.parquet


[2026-05-11 06:50:11] [INFO] src.indexing: [article_level] BM25 index already exists — loading from cache.
INFO:src.indexing:[article_level] BM25 index already exists — loading from cache.


Chunks loaded: 9,999


[2026-05-11 06:50:12] [INFO] src.indexing: [article_level] Loaded: 9,999 chunks.
INFO:src.indexing:[article_level] Loaded: 9,999 chunks.



Build stats:
{
  "strategy": "article_level",
  "total_chunks": 9999,
  "vocab_size": 27943,
  "avg_doc_length_tokens": 404.6,
  "tokenization_seconds": 2.6,
  "build_seconds": 2.5,
  "tokenizer": "whitespace_vi",
  "bm25_variant": "BM25Okapi"
}

✅ article_level BM25 ready — 9,999 chunks, avgdl=404.6 tokens


## Cell 6 — Verify All Indexes

Checks that for every strategy:
- Both `bm25_index.pkl` and `chunk_ids.json` exist
- BM25 corpus length == len(chunk_ids) (alignment guarantee for Phase 6)

In [7]:
from src.indexing import verify_bm25_index
import os

print("Verifying BM25 indexes...\n")
all_ok = True
for strategy in strategies:
    strategy_dir = os.path.join(bm25_base, strategy)
    pkl_path = os.path.join(strategy_dir, 'bm25_index.pkl')
    ids_path = os.path.join(strategy_dir, 'chunk_ids.json')
    stats_path = os.path.join(strategy_dir, 'build_stats.json')

    # File existence check
    files_ok = all(os.path.exists(p) for p in [pkl_path, ids_path])
    if not files_ok:
        print(f"  ❌ [{strategy}] Missing output files")
        all_ok = False
        continue

    # Alignment check
    try:
        ok = verify_bm25_index(bm25_base, strategy)
        ids  = json.load(open(ids_path))
        size = os.path.getsize(pkl_path) / 1e6
        print(f"  ✅ [{strategy}] {len(ids):,} chunks  |  pkl={size:.1f} MB")
        if os.path.exists(stats_path):
            s = json.load(open(stats_path))
            print(f"       vocab={s.get('vocab_size','?'):,}  "
                  f"avgdl={s.get('avg_doc_length_tokens','?')} tokens  "
                  f"build={s.get('build_seconds','?')}s")
    except AssertionError as e:
        print(f"  ❌ [{strategy}] {e}")
        all_ok = False

print()
if all_ok:
    print("✅ All 3 BM25 indexes verified. Phase 4 complete.")
    print("Confirm 'Xong' before proceeding to Phase 5 (Synthetic QA Generation).")
else:
    print("❌ One or more indexes failed verification. Re-run the failing cell(s).")

Verifying BM25 indexes...



[2026-05-11 06:50:15] [INFO] src.indexing: [fixed_size] BM25 index verified — 45,764 chunks, avgdl=206.0 tokens
INFO:src.indexing:[fixed_size] BM25 index verified — 45,764 chunks, avgdl=206.0 tokens


  ✅ [fixed_size] 45,764 chunks  |  pkl=59.1 MB
       vocab=37,546  avgdl=206.0 tokens  build=4.3s


[2026-05-11 06:50:19] [INFO] src.indexing: [sentence_aware] BM25 index verified — 64,197 chunks, avgdl=153.4 tokens
INFO:src.indexing:[sentence_aware] BM25 index verified — 64,197 chunks, avgdl=153.4 tokens


  ✅ [sentence_aware] 64,197 chunks  |  pkl=64.9 MB
       vocab=36,957  avgdl=153.4 tokens  build=5.7s


[2026-05-11 06:50:20] [INFO] src.indexing: [article_level] BM25 index verified — 9,999 chunks, avgdl=404.6 tokens
INFO:src.indexing:[article_level] BM25 index verified — 9,999 chunks, avgdl=404.6 tokens


  ✅ [article_level] 9,999 chunks  |  pkl=20.6 MB
       vocab=27,943  avgdl=404.6 tokens  build=2.5s

✅ All 3 BM25 indexes verified. Phase 4 complete.
Confirm 'Xong' before proceeding to Phase 5 (Synthetic QA Generation).


## Cell 7 — Smoke Test: BM25 Search

Run a sample query against the `fixed_size` index to confirm the index
is queryable and returns plausible results.

In [8]:
from src.indexing import load_bm25_index, tokenize_vi
import numpy as np

# Load the fixed_size index (already built above)
bm25_test, ids_test = load_bm25_index(bm25_base, 'fixed_size')
df_meta = pd.read_parquet(os.path.join(chunks_base, 'fixed_size', 'chunks.parquet'))
df_meta = df_meta.reset_index(drop=True)

TEST_QUERY = "lãi suất ngân hàng Việt Nam 2023"
query_tokens = tokenize_vi(TEST_QUERY)
scores = bm25_test.get_scores(query_tokens)
top_k = 5
top_indices = np.argsort(scores)[-top_k:][::-1]

print(f"Query : {TEST_QUERY!r}")
print(f"Tokens: {query_tokens}")
print(f"\nTop-{top_k} BM25 results:")
for rank, idx in enumerate(top_indices, 1):
    chunk_id = ids_test[idx]
    score    = scores[idx]
    row      = df_meta.iloc[idx]
    print(f"  #{rank}  score={score:.4f}  chunk_id={chunk_id}")
    print(f"       title : {str(row['title'])[:70]}...")
    print(f"       year  : {row['year']}  source: {row['source']}")
    print()

print("✅ Smoke test passed.")

[2026-05-11 06:50:22] [INFO] src.indexing: [fixed_size] BM25 index loaded — 45,764 chunks, avgdl=206.0
INFO:src.indexing:[fixed_size] BM25 index loaded — 45,764 chunks, avgdl=206.0


Query : 'lãi suất ngân hàng Việt Nam 2023'
Tokens: ['lãi', 'suất', 'ngân', 'hàng', 'việt', 'nam', '2023']

Top-5 BM25 results:
  #1  score=15.9794  chunk_id=d177cb7f027815fa_c0001
       title : Ước tính Ngân hàng Nhà nước đã mua vào 3,6 tỷ USD kể từ đầu năm 2023?...
       year  : 2023  source: vneconomy.vn

  #2  score=15.8898  chunk_id=a9873df4ffb00640_c0000
       title : Kỳ vọng lãi suất điều hành giảm thêm 50 điểm cơ bản trong quý III/2023...
       year  : 2023  source: baodautu.vn

  #3  score=15.4918  chunk_id=ce887438b5a8b25f_c0004
       title : Lãi suất tiết kiệm ngân hàng nào cao nhất tháng 5/2023?...
       year  : 2023  source: vneconomy.vn

  #4  score=15.3245  chunk_id=5c6861d951623321_c0000
       title : Bản Việt ưu đãi lãi suất cho vay, chỉ 10.5%/năm...
       year  : 2023  source: vneconomy.vn

  #5  score=15.1710  chunk_id=df02e7cbcbcec5f3_c0001
       title : Người vay mua nhà mắc kẹt trong vòng xoáy lãi suất tăng cao...
       year  : 2023  source: baodautu.vn

